## INGESTINATION 

### PART A: Divide our documents into chunks

In [1]:
import os
import glob
import tiktoken
import numpy as np
import frontmatter
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 9 files in the knowledge base
Total characters in knowledge base: 41,776


In [4]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 9,001


In [5]:
# Load every markdown file. The YAML front matter (between the --- lines) becomes
# metadata, and only the body text is kept as the page content.

documents = []
for file_path in files:
    post = frontmatter.load(file_path, encoding="utf-8")
    # Chroma only accepts str/int/float/bool metadata, so turn lists into "a, b, c"
    metadata = {
        key: ", ".join(value) if isinstance(value, list) else value
        for key, value in post.metadata.items()
    }
    metadata["doc_type"] = post.metadata.get("type", "general")
    metadata["source"] = file_path
    documents.append(Document(page_content=post.content, metadata=metadata))

print(f"Loaded {len(documents)} documents")

Loaded 9 documents


In [6]:
documents[0]

Document(metadata={'type': 'career', 'name': "What I'm Looking For", 'topics': 'roles, internship, part-time, location, career-direction', 'doc_type': 'career', 'source': 'knowledge-base\\career.md'}, page_content="# What I'm Looking For: My Career Goals and the Roles I Want\n\n## My career direction\n\nIn my career search, I am building my career around artificial intelligence, with a focus on LLMs, RAG, and agentic AI. I want to work directly on AI systems and keep developing my practical engineering skills by building real products and solving real problems. I am most interested in systems that use LLMs as part of real applications, not in working with AI only at a theoretical level.\n\n## The roles I'm looking for\n\nIn my career search, I am interested in these roles: LLM Engineer, AI Engineer, ML Engineer, AI/ML Intern, GenAI/LLM Intern, and Junior AI/ML Engineer. I am open to internships, part-time roles, trainee positions, and junior-level positions.\n\n## Why I'm looking for i

In [7]:
# Divide into chunks in two stages:
#   1. MarkdownHeaderTextSplitter cuts each file at its # / ## headers, so every section
#      becomes one chunk (strip_headers=False keeps the header text inside the chunk).
#   2. RecursiveCharacterTextSplitter is a safety net: it only cuts sections that are
#      longer than chunk_size, preferring paragraph breaks, then sentences.

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2")],
    strip_headers=False,
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

sections = []
for doc in documents:
    for section in header_splitter.split_text(doc.page_content):
        # keep the file's metadata and add the section's own headers (h1, h2)
        section.metadata = {**doc.metadata, **section.metadata}
        sections.append(section)

chunks = text_splitter.split_documents(sections)

# A chunk cut from the middle of a long section has lost its header (and so the topic
# name), so put the header back before embedding.
for chunk in chunks:
    if not chunk.page_content.startswith("#"):
        chunk.page_content = f"## {chunk.metadata.get('h2', '')} (continued)\n{chunk.page_content}"

print(f"Divided into {len(sections)} sections and {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 112 sections and 114 chunks
First chunk:

page_content='# What I'm Looking For: My Career Goals and the Roles I Want  
## My career direction  
In my career search, I am building my career around artificial intelligence, with a focus on LLMs, RAG, and agentic AI. I want to work directly on AI systems and keep developing my practical engineering skills by building real products and solving real problems. I am most interested in systems that use LLMs as part of real applications, not in working with AI only at a theoretical level.' metadata={'type': 'career', 'name': "What I'm Looking For", 'topics': 'roles, internship, part-time, location, career-direction', 'doc_type': 'career', 'source': 'knowledge-base\\career.md', 'h1': "What I'm Looking For: My Career Goals and the Roles I Want", 'h2': 'My career direction'}


In [8]:
chunks[100]

Document(metadata={'type': 'project', 'name': 'Fifa_Box', 'topics': 'rag, hybrid-retrieval, bm25, reranking, evaluation, desktop-app', 'stack': 'Python, ChromaDB, BM25, OpenAI, LiteLLM, cross-encoder, pywebview, HTML/CSS/JavaScript', 'doc_type': 'project', 'source': 'knowledge-base\\projects\\fifa-box.md', 'h1': 'Fifa_Box: Hybrid RAG Assistant for FIFA World Cup History', 'h2': 'Technologies used in Fifa_Box'}, page_content='## Technologies used in Fifa_Box  \n- Retrieval: ChromaDB, OpenAI `text-embedding-3-large`, BM25 (`rank_bm25`), cross-encoder reranking (`ms-marco-MiniLM-L-6-v2`)\n- LLM: OpenAI models, LiteLLM\n- Application: Python, pywebview, HTML, CSS, JavaScript\n- Supporting libraries: Pydantic, python-dotenv')

### PART B: Make vectors and store in Chroma

In [9]:
# Pick an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 114 documents


In [10]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 114 vectors with 3,072 dimensions in the vector store


### Part C: Visualize!

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()